# Recreating Original Moisture and Density Calculations #
Author: Maddox Chastain

Summer 2026

This notebook has the purpose of recreating the calculations seen within the original MAD (moisture and density) spreadsheets for all study sites of IODP EXP501. By recreating these original calculations, I hope to be able to easily adjust them to the varied levels of salinity seen at each site.

- when correcting for varied salinity, keep an eye out for any equations using the standard lab values for pore-water salinity (s = 0.035 = 35/1000) and pore-water density (rho_pw = 1.024). These are the values we will have to focus on correcting using the soon-to-be-developed interpolated salinity model!!!

- also need to watch for usage of the calculated average seawater salt density value, rho_salt = 2.20 g/cm^3 (Lyman and Fleming, 1940; Weast et al., 1985)
    - while the paper I was given says that the most common density of salt used for MAD is 2.20 g/cm^3, the excel sheets show a used value of 2.22 g/cm^3, so I used that instead in the calculations below! 
    - according to the paper, "the average density of the standard seawater salt is between 2.10 and 2.24 g/cm^3." (section 2, page 4. PP Handbook, Peter Blum, November 1997.), so this variation in salt density isn't anything super out of the ordinary. 

- volume/weight grains = volume/weight solids when referencing PP Handbook (I think)

In [165]:
import pandas as pd

### Reading in Data ###

To recreate the original MAD calculations done for each site, I'll read in the first 20 columns of data from the combined MAD spreadsheet using Pandas. 

After reading in this initial data, I'll recreate the original calculations using the provided equations. Pandas allows for simple creation of new columns based upon arithmetic manipulations of existing columns, so this process is actually pretty simple! Yay!

In [166]:
df = pd.read_csv('/mnt/c/Users/maddo/onedrive/Desktop/Work stuff/SURF 2026/MAD-FF-Corrections-from-Varied-Salinity/spreadsheets/03_MAD/COMBINED_MAD/MAD_DATA_CSV.csv')

### Obtaining Beaker Volumes ###

In [167]:
bkrs_df = pd.read_csv('/mnt/c/Users/maddo/onedrive/Desktop/Work stuff/SURF 2026/MAD-FF-Corrections-from-Varied-Salinity/spreadsheets/03_MAD/COMBINED_MAD/beakers_CSV.csv')#, index_col = 'Number')

In [168]:
for row in range(0, 600):
    num = df.iat[row, 12]
    for line in range(0, 137):
        if bkrs_df.iat[line, 0] == num:
            value = bkrs_df.iat[line, 2]
            df.at[row, 'Beaker Volume, cc'] = value
        else:
            continue

### Weight Calculations ###

In [169]:
full_wet_weight = df['WET Weight (Sample + Beaker), g ']
full_dry_weight = df['DRY Weight (Sample + Beaker), g']
beaker_weight = df['Beaker Weight,         g']


# WEIGHT WET Sample, g (full wet weight - beaker weight)
df['WEIGHT WET Sample, g'] = full_wet_weight - beaker_weight
wet_weight = df['WEIGHT WET Sample, g']

# WEIGHT DRY Sample, g (full dry weight - breaker weight)
df['WEIGHT DRY Sample, g'] = full_dry_weight - beaker_weight
dry_weight = df['WEIGHT DRY Sample, g']

# WEIGHT Pore Water, g ( (wet weight - dry weight)/(1-s), where s = 35/1000 or 0.035 )
df['WEIGHT Pore Water, g'] = (wet_weight - dry_weight)/(1-35/1000)

# WEIGHT Salt, g (weight pore water - (wet weight - dry weight))
df['WEIGHT Salt, g'] = (df['WEIGHT Pore Water, g']) - (wet_weight - dry_weight)

# WEIGHT Grains, g (weight dry - weight salt) <- this is how it's done in the spreadsheets
# This one keeps giving me problems, so I went with an eqn from the handbook instead (found in section 2 on page 13)
# Still giving me problems tho :(
df['WEIGHT Grains, g'] = (dry_weight - 0.035*wet_weight)/(1-0.035)


In [170]:
# checking progress made in data frame
df.head(n=3)

,Exp,Site,Hole,Core,Type,Section,"Sample Top, cm","Sample Bottom, cm",Combined id,Sample Label,...,"DRY Weight (Sample + Beaker), g",Scientist,Pycnometer Cell,"DRY Volume, cc","Beaker Volume, cc","WEIGHT WET Sample, g","WEIGHT DRY Sample, g","WEIGHT Pore Water, g","WEIGHT Salt, g","WEIGHT Grains, g"
0,501,M0111,A,1,H,1,63.5,65.5,M0111-A-1-1,"501-M0111A-1H-1,63.5-65.5",...,17.7360,Shintani,1,7.39638,4.5178,9.1205,7.6613,1.512124,0.052924,7.608376
1,501,M0111,A,2,H,2,38.0,40.0,M0111-A-2-2,"501-M0111A-2H-2,38-40",...,17.8746,Shintani,2,7.44070,4.4236,9.9534,8.0099,2.013990,0.070490,7.939410
2,501,M0111,A,3,H,1,38.0,40.0,M0111-A-3-1,"501-M0111A-3H-1,38-40",...,17.2814,Shintani,3,7.24460,4.5326,8.8712,7.1738,1.758964,0.061564,7.112236


### Volume Calculations ###

In [171]:
pw_weight = df['WEIGHT Pore Water, g']
salt_weight = df['WEIGHT Salt, g']
dry_volume = df['DRY Volume, cc']
beaker_volume = df['Beaker Volume, cc']

rho_pw = 1.024
rho_salt = 2.22


# VOLUME Pore Water, cc (pore water weight / pore water density) 
df['VOLUME Pore Water, cc'] = pw_weight / rho_pw

# VOLUME Salt, cc (weight salt / density salt)
df['VOLUME Salt, cc'] = salt_weight / rho_salt

# VOLUME Dry Sample, cc (dry volume - beaker volume)
df['VOLUME Dry Sample, cc'] = dry_volume - beaker_volume

# VOLUME Wet Sample, cc (dry sample volume - salt volume + pore water volume)
sample_dry_volume = df['VOLUME Dry Sample, cc']
salt_volume = df['VOLUME Salt, cc']
pw_volume = df['VOLUME Pore Water, cc']

df['VOLUME Wet Sample, cc'] = sample_dry_volume - salt_volume + pw_volume

# VOLUME Grains, cc (dry sample volume - salt volume)
df['VOLUME Grains, cc'] = sample_dry_volume - salt_volume


### Water Content, Density, Porosity, and Void Ratio Calculations ###

In [172]:
grain_weight = df['WEIGHT Grains, g']


# Water Content, (wet) v/v (weight pore water / weight wet sample)
df['Water Content, (wet) v/v'] = pw_weight / wet_weight

# Water Content, (dry) v/v (weight pore water / weight grains)
df['Water Content, (dry) v/v'] = pw_weight / grain_weight


In [173]:
sample_wet_volume = df['VOLUME Wet Sample, cc']
grain_weight = df['WEIGHT Grains, g']
grain_volume = df['VOLUME Grains, cc']


# Bulk Density, g/cc (weight wet sample / volume wet sample)
df['Bulk Density, g/cc'] = wet_weight / sample_wet_volume

# Dry Density, g/cc (weight salt / volume wet sample) <- eqn used in spreadsheets
# (weight grains / volume wet sample) <- eqn used in handbook
df['Dry Density, g/cc'] = grain_weight / sample_wet_volume

# Grain Density, g/cc (weight grains / volume grains)
df['Grain Density, g/cc'] = grain_weight / grain_volume


In [174]:


# Porosity, v/v (volume pore water / volume wet sample)
df['Porosity, v/v'] = pw_volume / sample_wet_volume

# Void Ratio, v/v (volume pore water / volume grains)
df['Void Ratio, v/v'] = pw_volume / grain_volume


In [175]:
# checking progress made in data frame
df.head(n=3)

,Exp,Site,Hole,Core,Type,Section,"Sample Top, cm","Sample Bottom, cm",Combined id,Sample Label,...,"VOLUME Dry Sample, cc","VOLUME Wet Sample, cc","VOLUME Grains, cc","Water Content, (wet) v/v","Water Content, (dry) v/v","Bulk Density, g/cc","Dry Density, g/cc","Grain Density, g/cc","Porosity, v/v","Void Ratio, v/v"
0,501,M0111,A,1,H,1,63.5,65.5,M0111-A-1-1,"501-M0111A-1H-1,63.5-65.5",...,2.87858,4.331424,2.854740,0.165794,0.198745,2.105658,1.756553,2.665173,0.340923,0.517274
1,501,M0111,A,2,H,2,38.0,40.0,M0111-A-2-2,"501-M0111A-2H-2,38-40",...,3.01710,4.952135,2.985348,0.202342,0.253670,2.009921,1.603230,2.659459,0.397159,0.658813
2,501,M0111,A,3,H,1,38.0,40.0,M0111-A-3-1,"501-M0111A-3H-1,38-40",...,2.71200,4.402007,2.684269,0.198278,0.247315,2.015263,1.615681,2.649599,0.390217,0.639928
